In [8]:
import torch, gc

from torch import nn
from torch.utils.data import Dataset, DataLoader

from model_utils import load_transformer_model, load_silence_latent, load_encoder, decode_latent_and_save_audio, \
    load_finetuning_audio_latents, get_files_in_path_as_array

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_dtype = torch.bfloat16
model_repo = "ACE-Step/acestep-v15-turbo-shift1"

cuda


In [3]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0530 14:37:21.377000 19708 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [4]:
path = "inputs/lora"
load_batch_size = 3 # due to VRAM constraints, can only load in limited batch sizes
train_data_limit = 10
files = get_files_in_path_as_array(path)
y = torch.Tensor().to(device).to(model_dtype)
with torch.no_grad():
    for start in range(0, min(len(files), train_data_limit), load_batch_size):
        end = min(start + load_batch_size, len(files))
        print(start, end)
        batch = load_finetuning_audio_latents(vae, files, device, model_dtype, start=start, end=end)
        y = torch.cat((y, batch), 0)
        del batch
        torch.cuda.empty_cache()
        gc.collect()
del vae

0 3
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
3 6
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
6 9
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
9 12
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
12 15
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
15 18
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
18 21
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
21 24
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])


In [6]:
print(y.shape)

torch.Size([24, 64, 1500])


In [171]:
class AudioLatentDataset(Dataset):
    def __init__(self, audio_latents):
        self.audio_latents = audio_latents

    def __len__(self):
        return self.audio_latents.shape[0]

    def __getitem__(self, item):
        return self.audio_latents[item]

train_dataset = AudioLatentDataset(y)
test_dataset = AudioLatentDataset(y)

In [186]:
batch_size = 1
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [10]:
dit = load_transformer_model(model_repo, model_dtype, device)

In [195]:
for parameter in dit.parameters():
    parameter.requires_grad = False

In [196]:
rank = 8
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.q_A = nn.Parameter(torch.randn(rank, layer.self_attn.q_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.q_proj.q_B = nn.Parameter(torch.randn(layer.self_attn.q_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_A = nn.Parameter(torch.randn(rank, layer.self_attn.v_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_B = nn.Parameter(torch.randn(layer.self_attn.v_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))


In [197]:
def lora_q_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.q_B, module.q_A)
    return outputs + torch.matmul(x, dW)

def lora_v_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.v_B, module.v_A)
    return outputs + torch.matmul(x, dW)

In [198]:
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.register_forward_hook(lora_q_forward_hook)
    layer.self_attn.v_proj.register_forward_hook(lora_v_forward_hook)

In [4]:
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

In [5]:
def do_inference(model, batch_size=1):
    text_hidden_states = torch.zeros(batch_size, 77, 1024, dtype=model_dtype, device=device)
    text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool, device=device)
    lyric_hidden_states = torch.zeros(batch_size, 123, 1024, dtype=model_dtype, device=device)
    lyric_attention_mask = torch.zeros(batch_size, 123, dtype=torch.bool, device=device)

    is_covers = torch.Tensor([False]).repeat(batch_size).to(device)

    seconds = 60
    infer_steps = 50
    frames_per_second = 25

    seq_len = int(seconds * frames_per_second)

    refer_audio_acoustic_hidden_states_packed = silence_latent[:, :, :1500].repeat(batch_size, 1, 1).permute(0, 2, 1)
    refer_audio_order_mask = torch.LongTensor(range(0, batch_size)).to(device)

    cur_chunk_mask = torch.ones(batch_size, seq_len, 64, dtype=torch.bool, device=device)
    cur_src_latents = silence_latent[:, :, :seq_len].repeat(batch_size, 1, 1).permute(0, 2, 1)

    print(lyric_hidden_states.shape)
    print(lyric_attention_mask.shape)
    print(text_hidden_states.shape)
    print(text_attention_mask.shape)
    print(is_covers.shape)
    print(cur_chunk_mask.shape)
    print(cur_src_latents.shape)
    print(refer_audio_acoustic_hidden_states_packed.shape)
    print(refer_audio_order_mask.shape)

    outputs = model.generate_audio(
        text_hidden_states=text_hidden_states,
        text_attention_mask=text_attention_mask,
        lyric_hidden_states=lyric_hidden_states,
        lyric_attention_mask=lyric_attention_mask,
        refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
        refer_audio_order_mask=refer_audio_order_mask,
        src_latents=cur_src_latents,
        chunk_masks=cur_chunk_mask,
        infer_steps=infer_steps,
        is_covers=is_covers,
        silence_latent=silence_latent,
        use_progress_bar=True,
        shift=1.0,

        repainting_start=torch.tensor([1.0]),
        repainting_end=torch.tensor([0.0]),
        audio_cover_strength=1.0,
        use_repainting=False
    )
    output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
    return output_latents

In [6]:
def decode_and_save(output_latents):
    vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)
    with torch.no_grad():
        decode_latent_and_save_audio(output_latents, vae, "lora")
    del vae
    torch.cuda.empty_cache()
    gc.collect()

In [11]:
output_latents = do_inference(dit, 2)
del dit
torch.cuda.empty_cache()
gc.collect()
decode_and_save(output_latents)

torch.Size([2, 123, 1024])
torch.Size([2, 123])
torch.Size([2, 77, 1024])
torch.Size([2, 77])
torch.Size([2])
torch.Size([2, 1500, 64])
torch.Size([2, 1500, 64])
torch.Size([2, 1500, 64])
torch.Size([2])


C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0531 17:28:40.432000 5380 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


torch.Size([2, 2, 2880000])
(2880000, 2)
(2880000, 2)


In [203]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, y in enumerate(dataloader):
        output_latents = do_inference(model, batch_size)

        print(output_latents.requires_grad)
        loss = loss_fn(output_latents, y)
        print(loss)
        print("Grad Fn:", loss.grad_fn)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loss, current = loss.item(), batch * batch_size
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    test_loss = 0

    with torch.no_grad():
        for y in dataloader:
            output_latents = do_inference(model, batch_size)
            test_loss += loss_fn(output_latents, y).item()

    test_loss /= num_batches
    print(f"Avg Test loss: {test_loss:>8f} \n")

In [204]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(dit.parameters())

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, dit, loss_fn, optimizer)
    test_loop(test_dataloader, dit, loss_fn)


Epoch 1
-------------------------------
False
tensor(2.5781, device='cuda:0', dtype=torch.bfloat16)
Grad Fn: None


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn